# English claim + evidence embeddings (CheckThat! 2026 Task 2)
### Qwen3-Embedding-8B · 4-bit NF4 · one 4096-d vector per row · resumable · Kaggle 2×T4

Every row of `train.json`, `validation.json` and `test.json` (no deduplication) becomes **one** text, the claim
followed by **all** its evidences, and is embedded into a single 4096-d vector:

```
Claim: <claim>

Evidence 1: <evidence 1>

Evidence 2: <evidence 2>
...
```

A row without evidence is embedded as `Claim: <claim>`. The settings match the existing `qwen3emb8b_en_v1` claim/trace
vectors except for the weights: raw text (no instruction prefix), last-token pooling, L2-normalised, at most 8192 tokens.
Longer texts lose the **end of the evidence**; the claim is always kept. The model weights are loaded in 4-bit
(bitsandbytes NF4, double quantisation, fp16 compute), so each T4 holds its own copy and the two GPUs work in parallel.

The reasoning-trace vectors are **not** recomputed. `mlp-english-top5-ranked-claim-evidence` reads them from the
existing `{train,val,test}_qwen3emb8b_en_v1.npz` and uses the vectors of this notebook in place of the claim vector.

**Outputs** (in `OUTPUT_DIR`), one file per split, `{split}_claimevid_qwen3emb8b4bit_en_v1.npz`:

| key | shape | content |
|:--|:--|:--|
| `emb` | (N, 4096) float32 | claim+evidence vectors, row `r` belongs to `__ids__[r]` |
| `__ids__` | (N,) str | position in the file (train/val) or `query_id` (test), sorted |
| `ntok` | (N,) int32 | tokens fed to the model (after truncation, incl. end token) |
| `truncated` | (N,) bool | text was longer than `MAX_SEQ_LENGTH` |
| `__meta__` | str | JSON of every embedding setting (checked on every reuse) |

**Resume.** Work is saved in shards of 64 rows as soon as each shard is done. On start, every merged file or shard
with the same settings found in `OUTPUT_DIR` or under `RESUME_SEARCH_ROOT` is checked and reused, and only missing rows
are embedded. If all three merged files exist, the model is not even loaded. The run stops cleanly after
`EMBED_TIME_BUDGET_HOURS`. To continue, save the version, add its output as an input, and run again.

**Run on Kaggle:** accelerator **GPU T4 ×2**, **internet on** (model download, `bitsandbytes` install if missing).
Check the paths in §1, then *Save & Run All*.

## 1. Configuration

In [ ]:
# ─────────────────────────── INPUT / OUTPUT PATHS ───────────────────────────
DATA_DIR           = "/kaggle/input/datasets/d202511054/clef-2026-checkthat-task-2-data/English"   # train.json, validation.json, test.json
OUTPUT_DIR         = "/kaggle/working"
RESUME_SEARCH_ROOT = "/kaggle/input"      # searched recursively for earlier shards / merged files with the same EMB_TAG
FP16_EMB_DIR       = "/kaggle/input/datasets/d202511054/qwen3-en-embeddings"   # optional, existing qwen3emb8b_en_v1 npz,
                                                                               # only for sanity checks; None = skip them

# ─────────────────────────── EMBEDDING SETTINGS ───────────────────────────
MODEL_ID        = "Qwen/Qwen3-Embedding-8B"
EMB_TAG         = "claimevid_qwen3emb8b4bit_en_v1"   # file tag: change it if you change any setting in this block
MAX_SEQ_LENGTH  = 8192
INSTRUCTION     = None                     # None = raw text (same as the existing claim / trace vectors)
CLAIM_PREFIX    = "Claim: "
EVIDENCE_PREFIX = "Evidence {i}: "         # i = 1, 2, ...
SEPARATOR       = "\n\n"
COMPUTE_DTYPE   = "float16"                # T4 has no bf16; NF4 weights are de-quantised to this dtype
STORE_DTYPE     = "float32"

# ─────────────────────────── THROUGHPUT / RESUME ───────────────────────────
N_REPLICAS              = 2                # model copies, one per GPU (4-bit 8B ≈ 5.5 GB)
TOKEN_BUDGET            = 16384            # max padded tokens (batch × longest) per forward pass, per GPU
MAX_TEXTS_PER_BATCH     = 32
SHARD_SIZE              = 64               # rows per resumable shard
EMBED_TIME_BUDGET_HOURS = 11.0             # no new shard is started after this (Kaggle limit is 12 h)
DELETE_SHARDS_AFTER_MERGE = True
SANITY_ROWS             = 300              # rows per split for the alignment check in §8

# ─────────────────────────── DATA FIELDS ───────────────────────────
CLAIM_FIELD    = "claim"
EVIDENCE_FIELD = "evidences"
TEST_ID_FIELD  = "query_id"
TRACE_EMB_TAG  = "qwen3emb8b_en_v1"        # tag of the existing claim/trace npz (sanity checks only)

SHARD_ROOT = f"{OUTPUT_DIR}/emb_shards/{EMB_TAG}"
EMB_META = {
    "tag": EMB_TAG, "model": MODEL_ID, "quantization": "bitsandbytes nf4, double quant",
    "compute_dtype": COMPUTE_DTYPE, "max_seq_length": MAX_SEQ_LENGTH, "instruction": INSTRUCTION,
    "template": {"claim": CLAIM_PREFIX, "evidence": EVIDENCE_PREFIX, "separator": SEPARATOR},
    "evidence_field": EVIDENCE_FIELD, "pooling": "last_token", "l2_normalized": True,
    "truncation": "right (end of the evidence), end token kept", "store_dtype": STORE_DTYPE,
}

## 2. Setup

In [ ]:
import os, re, gc, sys, json, glob, math, time, queue, shutil, threading, warnings, subprocess, importlib.util
from collections import OrderedDict

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
T_START = time.time()
os.makedirs(SHARD_ROOT, exist_ok=True)

print(f"PyTorch : {torch.__version__} | CUDA {torch.cuda.is_available()}")
for _i in range(torch.cuda.device_count()):
    _p = torch.cuda.get_device_properties(_i)
    print(f"  GPU {_i}: {_p.name}  {_p.total_memory / 1e9:.1f} GB")
print(json.dumps(EMB_META, indent=2, ensure_ascii=False))

## 3. Load the data and build the texts
IDs: train and validation use the position in the file (0..N-1); test uses `query_id`, the same keys as the
existing embedding files. Rows are stored sorted by ID. All rows are kept, including repeated claims.

In [ ]:
SPLIT_FILES = OrderedDict(train="train.json", val="validation.json", test="test.json")

def find_one(root, rel):
    """Shortest path under `root` ending in `rel`; Arabic / Spanish folders are ignored."""
    if not root or not os.path.isdir(root):
        return None
    hits = [h for h in glob.glob(os.path.join(root, "**", rel), recursive=True)
            if "arab" not in h.lower() and "span" not in h.lower()]
    return min(hits, key=len) if hits else None

def load_json(path):
    with open(path, encoding="utf-8") as f:
        s = f.read()
    try:
        return json.loads(s)
    except json.JSONDecodeError:
        # some copies of English train.json have one entry whose '"Reasoning_traces": [' key is missing
        print(f"  {os.path.basename(path)}: invalid JSON, repairing the missing 'Reasoning_traces' key")
        return json.loads(re.sub(r'\{\s*\nx?(\s+)"(?![A-Za-z_]+"\s*:)', r'{\n        "Reasoning_traces": [\n\1"', s))

def evidence_list(item):
    ev = item.get(EVIDENCE_FIELD) or []
    if isinstance(ev, str):
        ev = [ev]
    return [str(e).strip() for e in ev if str(e).strip()]

def build_text(item):
    parts = [CLAIM_PREFIX + item[CLAIM_FIELD].strip()]
    parts += [EVIDENCE_PREFIX.format(i=i) + e for i, e in enumerate(evidence_list(item), 1)]
    text = SEPARATOR.join(parts)
    return text if INSTRUCTION is None else f"Instruct: {INSTRUCTION}\nQuery:{text}"

SPLITS, stats = OrderedDict(), {}
for split, fname in SPLIT_FILES.items():
    path = find_one(DATA_DIR, fname)
    if path is None:
        raise FileNotFoundError(f"{fname} not found under {DATA_DIR}: attach the data and fix DATA_DIR in §1")
    items = load_json(path)
    for i, it in enumerate(items):
        if not isinstance(it.get(CLAIM_FIELD), str):
            raise ValueError(f"{split} row {i}: '{CLAIM_FIELD}' missing or not a string")
        if not isinstance(it.get(EVIDENCE_FIELD, []), (list, str, type(None))):
            raise ValueError(f"{split} row {i}: '{EVIDENCE_FIELD}' must be a list of strings")
    ids = [it[TEST_ID_FIELD] for it in items] if split == "test" else list(range(len(items)))
    if len(set(ids)) != len(ids):
        raise ValueError(f"{split}: duplicate ids")
    by_id = dict(zip(ids, items))
    order = sorted(ids)
    text = {i: build_text(by_id[i]) for i in order}
    n_ev = np.array([len(evidence_list(by_id[i])) for i in order])
    chars = np.array([len(text[i]) for i in order])
    SPLITS[split] = {"path": path, "order": order, "text": text, "n_evid": n_ev}
    stats[split] = {"rows": len(order), "rows without evidence": int((n_ev == 0).sum()),
                    "rows without the evidence key": sum(EVIDENCE_FIELD not in it for it in items),
                    "evidences min/mean/max": f"{n_ev.min()}/{n_ev.mean():.2f}/{n_ev.max()}",
                    "text chars p50": int(np.median(chars)), "text chars p90": int(np.percentile(chars, 90)),
                    "text chars max": int(chars.max())}
    print(f"{split:5s}: {path}")
display(pd.DataFrame(stats))

_ex = SPLITS["test"]["text"][SPLITS["test"]["order"][0]]
print(f"\nExample (test, first {600} of {len(_ex)} chars):\n{_ex[:600]}...")

## 4. Shards, merged files and resume
A shard or merged file is reused only if its `__meta__` equals `EMB_META`, its IDs are exactly the expected ones, and
all its vectors are finite and unit-norm. Anything else is ignored and recomputed.

In [ ]:
def shard_plan(split):
    o = SPLITS[split]["order"]
    return [(k, o[k * SHARD_SIZE:(k + 1) * SHARD_SIZE]) for k in range(math.ceil(len(o) / SHARD_SIZE))]

def shard_path(split, k):  return f"{SHARD_ROOT}/{split}/shard_{k:05d}.npz"
def final_name(split):     return f"{split}_{EMB_TAG}.npz"
def final_path(split):     return f"{OUTPUT_DIR}/{final_name(split)}"

def read_checked(path, ids):
    """Arrays of a shard / merged file if it matches EMB_META and `ids` exactly, else None."""
    if not os.path.exists(path):
        return None
    try:
        with np.load(path, allow_pickle=False) as z:
            if "__meta__" not in z.files or json.loads(str(z["__meta__"])) != EMB_META:
                return None
            if list(z["__ids__"]) != [str(i) for i in ids]:
                return None
            emb = z["emb"]
            if emb.ndim != 2 or emb.shape[0] != len(ids) or not np.isfinite(emb).all():
                return None
            if np.abs(np.linalg.norm(emb.astype(np.float32), axis=1) - 1).max() > 1e-3:
                return None
            return {"emb": emb, "ntok": z["ntok"], "truncated": z["truncated"]}
    except Exception as e:
        print(f"  ! unreadable {path}: {e}")
        return None

def save_atomic(path, ids, emb, ntok, truncated):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path[:-4] + ".tmp.npz"
    np.savez(tmp, __meta__=np.array(json.dumps(EMB_META, ensure_ascii=False)),
             __ids__=np.array([str(i) for i in ids]), emb=emb.astype(STORE_DTYPE),
             ntok=np.asarray(ntok, np.int32), truncated=np.asarray(truncated, bool))
    os.replace(tmp, path)

def import_previous():
    """Reuses merged files / shards from OUTPUT_DIR or RESUME_SEARCH_ROOT (copied into OUTPUT_DIR)."""
    final, copied = {}, 0
    search = bool(RESUME_SEARCH_ROOT) and os.path.isdir(RESUME_SEARCH_ROOT)
    for split in SPLITS:
        order = SPLITS[split]["order"]
        cands = [final_path(split)]
        if search:
            cands += sorted(glob.glob(f"{RESUME_SEARCH_ROOT}/**/{final_name(split)}", recursive=True), key=len)
        for p in cands:
            if os.path.exists(p) and read_checked(p, order) is not None:
                if os.path.abspath(p) != os.path.abspath(final_path(split)):
                    shutil.copy(p, final_path(split))
                    print(f"[{split}] merged file reused ← {p}")
                final[split] = final_path(split)
                break
        if split in final or not search:
            continue
        plan = dict(shard_plan(split))
        for p in glob.glob(f"{RESUME_SEARCH_ROOT}/**/emb_shards/{EMB_TAG}/{split}/shard_*.npz", recursive=True):
            m = re.search(r"shard_(\d+)\.npz$", p)
            k = int(m.group(1)) if m else -1
            dst = shard_path(split, k)
            if k in plan and not os.path.exists(dst) and read_checked(p, plan[k]) is not None:
                os.makedirs(os.path.dirname(dst), exist_ok=True)
                shutil.copy(p, dst)
                copied += 1
    print(f"shards imported from {RESUME_SEARCH_ROOT}: {copied}")
    return final

FINAL = import_previous()
PENDING, status = [], {}
for split in SPLITS:
    if split in FINAL:
        status[split] = "merged (reused)"
        continue
    plan = shard_plan(split)
    todo = [(split, k, ids) for k, ids in plan if read_checked(shard_path(split, k), ids) is None]
    PENDING += todo
    status[split] = f"{len(plan) - len(todo)}/{len(plan)} shards done"
print("status:", status)
print(f"shards to embed: {len(PENDING)} ({sum(len(j[2]) for j in PENDING)} rows)")

## 5. Load the 4-bit model (skipped when nothing is pending)
One model copy per GPU, each with its own tokenizer, so the two GPUs can run in separate threads. The tokenizer
of Qwen3-Embedding appends `<|endoftext|>`. That end token is kept after truncation and is the pooled
(last) position. Padding is on the left.

In [ ]:
REPLICAS, EOS_SUFFIX, PAD_ID, EMB_DIM = [], None, None, None

def detect_suffix(tok):
    """Token(s) the tokenizer appends at the end (official Qwen3-Embedding: <|endoftext|>); else append it ourselves."""
    probe = "Claim: test 123"
    with_sp = tok(probe)["input_ids"]
    plain = tok(probe, add_special_tokens=False)["input_ids"]
    if len(with_sp) > len(plain) and with_sp[:len(plain)] == plain:
        return with_sp[len(plain):]
    end = tok.get_vocab().get("<|endoftext|>", tok.eos_token_id)
    if end is None:
        raise ValueError("the tokenizer appends no end token and has no <|endoftext|>/eos: cannot last-token pool")
    return [end]

def tokenize(tok, texts):
    """Token ids with the end token, truncated on the right to MAX_SEQ_LENGTH; plus a truncation flag per text."""
    keep = MAX_SEQ_LENGTH - len(EOS_SUFFIX)
    plain = tok(texts, add_special_tokens=False)["input_ids"]
    return [p[:keep] + EOS_SUFFIX for p in plain], [len(p) > keep for p in plain]

def make_batches(lengths):
    order = sorted(range(len(lengths)), key=lambda i: -lengths[i])   # longest first: an OOM shows up at once
    batches, cur, cur_max = [], [], 0
    for i in order:
        new_max = max(cur_max, lengths[i])
        if cur and (new_max * (len(cur) + 1) > TOKEN_BUDGET or len(cur) >= MAX_TEXTS_PER_BATCH):
            batches.append(cur)
            cur, new_max = [], lengths[i]
        cur.append(i)
        cur_max = new_max
    if cur:
        batches.append(cur)
    return batches

class Replica:
    def __init__(self, gpu):
        from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig
        self.gpu, self.dev = gpu, torch.device(f"cuda:{gpu}")
        self.tok = AutoTokenizer.from_pretrained(MODEL_ID)
        dt = getattr(torch, COMPUTE_DTYPE)
        bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                 bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=dt)
        self.model = AutoModel.from_pretrained(MODEL_ID, quantization_config=bnb, device_map={"": gpu},
                                               attn_implementation="sdpa", low_cpu_mem_usage=True,
                                               **{DTYPE_KW: dt}).eval()

    @torch.inference_mode()
    def _forward(self, seqs):
        L = max(len(s) for s in seqs)
        ids = torch.full((len(seqs), L), PAD_ID, dtype=torch.long)
        mask = torch.zeros((len(seqs), L), dtype=torch.long)
        for r, s in enumerate(seqs):                     # left padding: last position = end token
            ids[r, L - len(s):] = torch.tensor(s)
            mask[r, L - len(s):] = 1
        h = self.model(input_ids=ids.to(self.dev), attention_mask=mask.to(self.dev),
                       use_cache=False).last_hidden_state[:, -1].float()
        finite = torch.isfinite(h).all(dim=1)
        return F.normalize(h, p=2, dim=-1).cpu().numpy(), finite.cpu().numpy()

    def encode(self, seqs):
        """OOM → split the batch in two; a non-finite row is re-encoded alone, and the run stops if it stays non-finite."""
        oom = False
        try:
            emb, finite = self._forward(seqs)
        except torch.cuda.OutOfMemoryError:
            oom = True
        if oom:
            torch.cuda.empty_cache()
            if len(seqs) == 1:
                raise RuntimeError(f"GPU {self.gpu}: OOM on one {len(seqs[0])}-token text; "
                                   f"lower MAX_SEQ_LENGTH or N_REPLICAS")
            h = len(seqs) // 2
            return np.concatenate([self.encode(seqs[:h]), self.encode(seqs[h:])])
        for j in np.where(~finite)[0]:
            e1, f1 = self._forward([seqs[j]])
            if not f1.all():
                raise RuntimeError(f"non-finite embedding for a {len(seqs[j])}-token text even when encoded alone "
                                   f"(fp16 overflow); set COMPUTE_DTYPE='float32' and change EMB_TAG")
            emb[j] = e1[0]
        return emb

    def embed_texts(self, texts):
        seqs, trunc = tokenize(self.tok, texts)
        out = np.empty((len(seqs), EMB_DIM), np.float32)
        for b in make_batches([len(s) for s in seqs]):
            out[b] = self.encode([seqs[i] for i in b])
        return out, np.array([len(s) for s in seqs], np.int32), np.array(trunc, bool)

if PENDING:
    if not torch.cuda.is_available():
        raise RuntimeError("no GPU: switch the accelerator to GPU T4 x2")
    if importlib.util.find_spec("bitsandbytes") is None:
        print("installing bitsandbytes ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "bitsandbytes"], check=True)
    import transformers
    from transformers import AutoTokenizer
    transformers.logging.set_verbosity_error()
    _v = tuple(int(x) for x in re.findall(r"\d+", transformers.__version__)[:2])
    DTYPE_KW = "dtype" if _v >= (4, 56) else "torch_dtype"
    print(f"transformers {transformers.__version__}")

    TOK = AutoTokenizer.from_pretrained(MODEL_ID)
    EOS_SUFFIX = detect_suffix(TOK)
    PAD_ID = TOK.pad_token_id if TOK.pad_token_id is not None else EOS_SUFFIX[-1]
    print(f"end token(s) {TOK.convert_ids_to_tokens(EOS_SUFFIX)} | pad id {PAD_ID}")

    for g in range(min(N_REPLICAS, torch.cuda.device_count())):
        t = time.time()
        REPLICAS.append(Replica(g))
        print(f"GPU {g}: model loaded in {time.time() - t:.0f}s | "
              f"{torch.cuda.memory_allocated(g) / 2**30:.1f} GiB allocated")
    EMB_DIM = REPLICAS[0].model.config.hidden_size
    print(f"{len(REPLICAS)} replica(s) | hidden size {EMB_DIM}")
else:
    print("nothing to embed: model not loaded")

## 6. Pre-flight: token lengths, speed, ETA and a 4-bit vs fp16 check
* Token lengths of all pending rows (how many are truncated at `MAX_SEQ_LENGTH`).
* The longest pending text is encoded alone (worst case for memory), then a batch of typical texts gives tokens/s and the ETA.
* **4-bit vs fp16:** 8 train claims are embedded alone, as raw text like the existing vectors, and compared with
  row 0 (the claim) of the existing fp16 `train_qwen3emb8b_en_v1.npz`. A cosine close to 1 means the 4-bit model
  reproduces the fp16 one.

In [ ]:
PENDING_TOK = 0
if PENDING:
    pend_texts = [SPLITS[s]["text"][i] for s, _, ids in PENDING for i in ids]
    lens = np.array([len(p) for p in TOK(pend_texts, add_special_tokens=False)["input_ids"]]) + len(EOS_SUFFIX)
    fed = np.minimum(lens, MAX_SEQ_LENGTH)
    PENDING_TOK = int(fed.sum())
    print(f"pending rows {len(lens)} | tokens p50 {int(np.median(lens))} | p90 {int(np.percentile(lens, 90))} | "
          f"max {lens.max()} | truncated at {MAX_SEQ_LENGTH}: {(lens > MAX_SEQ_LENGTH).sum()} | "
          f"tokens to embed {PENDING_TOK / 1e6:.2f} M")

    rep = REPLICAS[0]
    t = time.time()
    e, n, _ = rep.embed_texts([pend_texts[int(lens.argmax())]])
    print(f"longest text: {n[0]} tokens in {time.time() - t:.1f}s | norm {np.linalg.norm(e[0]):.4f} | "
          f"GPU 0 peak {torch.cuda.max_memory_allocated(0) / 2**30:.1f} GiB")

    rng = np.random.default_rng(0)
    sample = [pend_texts[i] for i in rng.choice(len(pend_texts), min(48, len(pend_texts)), replace=False)]
    torch.cuda.synchronize(0); t = time.time()
    e, n, _ = rep.embed_texts(sample)
    torch.cuda.synchronize(0); dt = time.time() - t
    tps = n.sum() / dt
    print(f"speed (1 GPU): {tps:,.0f} tokens/s on {len(sample)} random pending texts | finite {np.isfinite(e).all()}")
    print(f"ETA with {len(REPLICAS)} GPU(s): ~{PENDING_TOK / (tps * len(REPLICAS)) / 60:.0f} min "
          f"(time budget {EMBED_TIME_BUDGET_HOURS:.1f} h, {(time.time() - T_START) / 60:.0f} min used)")

    p16 = find_one(FP16_EMB_DIR, f"train_{TRACE_EMB_TAG}.npz")
    if p16:
        items = load_json(SPLITS["train"]["path"])[:8]
        e4, _, _ = rep.embed_texts([it[CLAIM_FIELD].strip() for it in items])
        with np.load(p16) as z:
            e16 = np.stack([z[f"id_{i}"][0].astype(np.float32) for i in range(len(items))])
        cos = (e4 * e16).sum(1) / np.linalg.norm(e16, axis=1)
        print(f"4-bit vs fp16 claim vectors (8 train claims): cos mean {cos.mean():.4f} | min {cos.min():.4f}")
        if cos.min() < 0.9:
            print("  ⚠ low agreement with the fp16 vectors: check the model / quantisation before using these")
    else:
        print(f"fp16 check skipped: train_{TRACE_EMB_TAG}.npz not found under {FP16_EMB_DIR}")

## 7. Embed the pending shards
Both GPUs take shards from one queue. Each shard is saved as soon as it is done, so an interrupted run loses at most
the shard in progress. No new shard starts after `EMBED_TIME_BUDGET_HOURS`.

In [ ]:
def run_embedding(jobs_list):
    jobs = queue.Queue()
    for j in jobs_list:
        jobs.put(j)
    stop, lock = threading.Event(), threading.Lock()
    st = {"done": 0, "tok": 0, "errors": [], "t0": time.time()}
    deadline = T_START + EMBED_TIME_BUDGET_HOURS * 3600

    def worker(rep):
        torch.cuda.set_device(rep.gpu)
        while not stop.is_set() and time.time() < deadline:
            try:
                split, k, ids = jobs.get_nowait()
            except queue.Empty:
                return
            t = time.time()
            try:
                emb, ntok, trunc = rep.embed_texts([SPLITS[split]["text"][i] for i in ids])
                save_atomic(shard_path(split, k), ids, emb, ntok, trunc)
            except Exception as e:
                with lock:
                    st["errors"].append(f"GPU {rep.gpu}, {split} shard {k}: {type(e).__name__}: {e}")
                stop.set()
                return
            with lock:
                st["done"] += 1
                st["tok"] += int(ntok.sum())
                rate = st["tok"] / (time.time() - st["t0"])
                eta = (PENDING_TOK - st["tok"]) / max(rate, 1) / 60
                print(f"[GPU {rep.gpu}] {split:5s} shard {k:4d} | {len(ids):3d} rows | {ntok.sum() / 1e3:7.1f}k tok | "
                      f"trunc {int(trunc.sum()):2d} | {time.time() - t:6.1f}s | {st['done']}/{len(jobs_list)} | "
                      f"{rate:,.0f} tok/s | ETA {eta:.0f} min", flush=True)

    threads = [threading.Thread(target=worker, args=(r,), daemon=True) for r in REPLICAS]
    for t in threads:
        t.start()
    try:
        for t in threads:
            t.join()
    except KeyboardInterrupt:
        stop.set()
        for t in threads:
            t.join()
        raise
    if st["errors"]:
        raise RuntimeError("embedding stopped:\n  " + "\n  ".join(st["errors"]))
    left = len(jobs_list) - st["done"]
    print(f"\n{st['done']} shard(s) embedded in {(time.time() - st['t0']) / 60:.1f} min" +
          (f" | {left} left: time budget reached. Save this version, add its output as an input, run again."
           if left else ""))

if PENDING:
    run_embedding(PENDING)
else:
    print("nothing to embed")

## 8. Merge, verify and sanity checks
A split is merged once all its shards exist. Each merged file is re-read and checked (meta, IDs, finite unit vectors).

**Alignment check** (needs `FP16_EMB_DIR`): for `SANITY_ROWS` rows, the cosine between the claim+evidence vector and
the existing claim-only vector of the **same** row should be clearly higher than with a **random other** row.
If both are about equal, the vectors are misaligned with the IDs.

In [ ]:
for split in SPLITS:
    if split in FINAL:
        continue
    plan = shard_plan(split)
    parts = [read_checked(shard_path(split, k), ids) for k, ids in plan]
    missing = sum(p is None for p in parts)
    if missing:
        print(f"[{split}] {missing}/{len(plan)} shards missing: not merged (run again to finish)")
        continue
    save_atomic(final_path(split), SPLITS[split]["order"],
                np.concatenate([p["emb"] for p in parts]), np.concatenate([p["ntok"] for p in parts]),
                np.concatenate([p["truncated"] for p in parts]))
    if read_checked(final_path(split), SPLITS[split]["order"]) is None:
        raise RuntimeError(f"[{split}] merged file failed verification")
    FINAL[split] = final_path(split)
    print(f"[{split}] merged → {final_path(split)} ({os.path.getsize(final_path(split)) / 1e6:.1f} MB)")
    if DELETE_SHARDS_AFTER_MERGE:
        shutil.rmtree(f"{SHARD_ROOT}/{split}", ignore_errors=True)

summary = {}
p16_paths = {s: find_one(FP16_EMB_DIR, f"{s}_{TRACE_EMB_TAG}.npz") for s in SPLITS}
for split, path in FINAL.items():
    order = SPLITS[split]["order"]
    a = read_checked(path, order)
    if a is None:
        raise RuntimeError(f"[{split}] {path} failed verification")
    nt, tr = a["ntok"], a["truncated"]
    row = {"rows": len(order), "dim": a["emb"].shape[1], "tokens p50": int(np.median(nt)),
           "tokens p90": int(np.percentile(nt, 90)), "tokens max": int(nt.max()), "truncated rows": int(tr.sum()),
           "rows without evidence": int((SPLITS[split]["n_evid"] == 0).sum())}
    if p16_paths[split]:
        rng = np.random.default_rng(0)
        pick = rng.choice(len(order), min(SANITY_ROWS, len(order)), replace=False)
        with np.load(p16_paths[split]) as z:
            c16 = np.stack([z[f"id_{order[r]}"][0].astype(np.float32) for r in pick])
        c16 /= np.linalg.norm(c16, axis=1, keepdims=True)
        ce = a["emb"][pick].astype(np.float32)
        row["cos same row"] = round(float((ce * c16).sum(1).mean()), 4)
        row["cos random row"] = round(float((ce * np.roll(c16, 1, axis=0)).sum(1).mean()), 4)
        if row["cos same row"] < row["cos random row"] + 0.1:
            print(f"  ⚠ {split}: claim+evidence vectors barely closer to their own claim than to a random one")
    summary[split] = row
display(pd.DataFrame(summary).astype(object))

with open(f"{OUTPUT_DIR}/{EMB_TAG}_meta.json", "w", encoding="utf-8") as f:
    json.dump({"meta": EMB_META, "files": FINAL, "data": {s: SPLITS[s]["path"] for s in SPLITS},
               "summary": summary}, f, indent=2, ensure_ascii=False, default=int)

if len(FINAL) == len(SPLITS):
    print(f"✓ all splits done: {list(FINAL.values())}")
else:
    print(f"✗ not finished: {[s for s in SPLITS if s not in FINAL]}. Save the version, add its output as an "
          f"input (it is found under RESUME_SEARCH_ROOT), and run again.")
print(f"total time {(time.time() - T_START) / 60:.1f} min")

In [ ]:
# free the GPUs
for r in REPLICAS:
    del r.model
REPLICAS = []
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()